⏬ 01 — Async IO Basics: The Event-Loop's Complete Toolkit

**Topic 12.03 | Async/Parallel Python — Part 3 of 4**

🌍 **Hook:** The download-boss's gallery: 50 files arrive every day. One by one →
all night long! Gather → wait on all 50 at the same time… but the server OWNER is angry —
"50 connections from one user?!" That's why there's **the LAW of LANES**: `Semaphore(3)` =
"only 3 downloads at a time" — fair usage + no chaos! 🚦

In this notebook, 4 heavier weapons of asyncio (built on the async/await foundation of the
previous topic!):
- 🐣 `create_task` — start now, take the result later (fire-and-collect!)
- 🐥 `Semaphore(n)` — lanes for downloads (flood-control 🌊)
- 🐔 per-task timeout + partial-success (some fail? the rest still win!)
- 🌐 the real world = `aiohttp` — a CONCEPT md-sample (offline sandbox here!)

## 🐣 Level 0 — `create_task`: Take the Order and Walk Off 📝

📖 **Definitions:**
- `gather(c1, c2)` = the waiter SITS and waits for all the orders together
- `create_task(coro)` = writes into the kitchen-ledger and LEAVES — the coroutine
  starts running in the background while you do the next task; `await task` whenever you need it.

🔍 **WHY?** tasks = a HANDLE to running-coroutines — you can cancel them, await
them, or poll them (`task.done()`). Use gather for a small squad; use
create_task for INDEPENDENT operations!

In [1]:
import asyncio, time
try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

def run(coro):
    return asyncio.run(coro)

async def kitchen(dish, t):
    await asyncio.sleep(t)
    print(f"  🍳 {dish} ready! ({t}s)")
    return dish

async def counter_order():
    t_dosa = asyncio.create_task(kitchen("dosa", 0.3))
    t_idli = asyncio.create_task(kitchen("idli", 0.15))
    print("📝 ordered both! (waiter aage badha — kaam chal raha...)")
    print("   dosa done?", t_dosa.done(), "(abhi nai — chal raha!)")
    r1 = await t_dosa
    print("   after await: done?", t_dosa.done(), "| got:", r1)
    r2 = await t_idli  # already hone wala (0.15 < 0.3) — instant!
    print("   idli:", r2, "(yeh toh pehle Hi ready tha — instant await!)")

run(counter_order())
# Expected OUTPUT:
# 📝 ordered both! (waiter aage badha — kaam chal raha...)
#    dosa done? False (abhi nai — chal raha!)
#   🍳 idli ready! (0.15s)
#   🍳 dosa ready! (0.3s)
#    after await: done? True | got: dosa
#    idli: idli (yeh toh pehle Hi ready tha — instant await!)

📝 ordered both! (waiter aage badha — kaam chal raha...)
   dosa done? False (abhi nai — chal raha!)
  🍳 idli ready! (0.15s)
  🍳 dosa ready! (0.3s)
   after await: done? True | got: dosa
   idli: idli (yeh toh pehle Hi ready tha — instant await!)


## 🐥 Level 1 — `Semaphore(2)`: 5 Downloads, ONLY 2 Lanes! 🚦

📖 **Definition:** `sem = asyncio.Semaphore(n)` + `async with sem:` =
at any ONE time a MAXIMUM of n coroutines are inside that section — the rest wait in line! 🔒

🔍 The courtroom story: 5 files in 2 lanes → 3 batches — time ≈ slowest+middle...
watch live: peak-concurrency is EXACTLY 2 (verify it with a counter!)

In [2]:
async def lane_demo():
    SEM = asyncio.Semaphore(2)
    live = 0
    peak = 0
    async def dl(fid, secs):
        nonlocal live, peak
        async with SEM:
            live += 1
            peak = max(peak, live)
            print(f"   ⏬ file-{fid} lane mil gayi (active={live})")
            await asyncio.sleep(secs)
            live -= 1
            return fid
    plan = [(0, 0.2), (1, 0.2), (2, 0.3), (3, 0.1), (4, 0.25)]
    t0 = time.perf_counter()
    got = await asyncio.gather(*(dl(i, s) for i, s in plan))
    el = time.perf_counter() - t0
    print(f"\n🏁 got: {got}")
    print(f"🚦 peak concurrency = {peak} (Semaphore(2) ka vada nibhaya!)")
    print(f"⏱️ {el:.2f}s — 2 lanes pe ~5×avg/2: na 5-parallel-0.3, na sequential-1.05!")

run(lane_demo())
# Expected OUTPUT (lane-acquire order deterministic!):
#    ⏬ file-0 lane mil gayi (active=1)
#    ⏬ file-1 lane mil gayi (active=2)
#    ⏬ file-2 lane mil gayi (active=1)   ← 0.2s pe f0 khalaa, f2 ko line mili
#    ⏬ file-3 lane mil gayi (active=2)   ← f1 bhi khatam, f3 lane me
#    ⏬ file-4 lane mil gayi (active=2)   ← 0.3s pe f3 (0.1s) done → f4
# 🏁 got: [0, 1, 2, 3, 4]
# 🚦 peak concurrency = 2 (Semaphore(2) ka vada nibhaya!)
# ⏱️ 0.55s (f4 0.3+0.25 = mehengi chain!)

   ⏬ file-0 lane mil gayi (active=1)
   ⏬ file-1 lane mil gayi (active=2)
   ⏬ file-2 lane mil gayi (active=1)
   ⏬ file-3 lane mil gayi (active=2)
   ⏬ file-4 lane mil gayi (active=2)

🏁 got: [0, 1, 2, 3, 4]
🚦 peak concurrency = 2 (Semaphore(2) ka vada nibhaya!)
⏱️ 0.55s — 2 lanes pe ~5×avg/2: na 5-parallel-0.3, na sequential-1.05!


## 🐔 Level 2 — Per-Task Timeout: Some Fail, the Rest Win! 🏆

📖 **Pattern:** news-wire — 4 "sources", each with its own `wait_for` timeout.
One slow source must not kill the whole digest — gather(return_exceptions=True)!

In [3]:
async def wire(source, t, timeout=0.22):
    try:
        async def fetch():
            await asyncio.sleep(t)
            return f"📰 {source} headlines"
        return await asyncio.wait_for(fetch(), timeout=timeout)
    except TimeoutError:
        return f"🛡️ {source} TIMEOUT (skip!)"

async def news_digest():
    feeds = [("Zeenews", 0.10), ("AajTak", 0.20), ("DDLazy", 0.50), ("OffersFact", 0.15)]
    print("📡 3 se zyada t waali atak jayengi (timeout=0.22)...")
    results = await asyncio.gather(*(wire(n, t) for n, t in feeds))
    for r in results:
        print("  ", r)
    ok = sum("📰" in r for r in results)
    print(f"🏆 digest: {ok}/4 sources — partial-success accepted!")

run(news_digest())
# Expected OUTPUT:
# 📡 3 se zyada t waali atak jayengi (timeout=0.22)...
#    📰 Zeenews headlines
#    📰 AajTak headlines
#    🛡️ DDLazy TIMEOUT (skip!)
#    📰 OffersFact headlines
# 🏆 digest: 3/4 sources — partial-success accepted!

📡 3 se zyada t waali atak jayengi (timeout=0.22)...
   📰 Zeenews headlines
   📰 AajTak headlines
   🛡️ DDLazy TIMEOUT (skip!)
   📰 OffersFact headlines
🏆 digest: 3/4 sources — partial-success accepted!


## 🌐 The Real World: `aiohttp` (CONCEPT — the sandbox here is offline!)

On real servers, downloads use the `aiohttp` library. Everything works exactly as you learned:
```python
import aiohttp, asyncio                      # pip install aiohttp

SEM = asyncio.Semaphore(3)                    # 3 lanes!

async def fetch(session, url):
    async with SEM:                           # flood-control
        async with session.get(url) as resp:  # async HTTP!
            return resp.status, await resp.text()  # await on network-IO

async def main():
    urls = ["https://api.github.com", "https://pypi.org/pypi/requests/json"]
    async with aiohttp.ClientSession() as s:
        reports = await asyncio.gather(*(fetch(s, u) for u in urls))
    print(reports)

asyncio.run(main())
```
🔍 You already know every move: async-with, SEM, gather, await-on-IO. `aiohttp` is
just this pattern on the real wire! (Offline-sandbox: the site won't load via curl here —
grasp the concept, then try it on your own machine!)

---
## ⚠️ 5 Classic Mistakes (IO-toolkit)
| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | Create a task but forget the await (killed by GC!) | every task is `await`ed/gather'ed |
| 2 | Forget the semicolon-brother of Semaphore (`with` miss) | `async with SEM:` always |
| 3 | Using Semaphore as a global and making things slow | understand its per-lane nature |
| 4 | Timeouts set too tight (everything fails) | per-task `wait_for` granularity |
| 5 | awaiting per-task in a gather loop (serial!) | make a list → one single gather |

## 📌 Recap — 30 Seconds
- `create_task` = take the order and move on 📝 · poll with `task.done()` · collect with `await`
- `Semaphore(n)` = n lanes 🚦 — verify deterministically with a peak-counter!
- per-task `wait_for` = partial-success digest 🏆 (one fail ≠ everything doomed)
- `aiohttp` = all of these patterns on a real server (grasp the concept offline, then try!)